# Spectral Analysis with Gaussian Fitting

This notebook performs spectral analysis on time-resolved spectroscopy data using multiple Gaussian models and a linear baseline.

## Overview
- Load processed spectral data from CSV
- Filter wavelengths (>=280 nm)
- Fit multiple Gaussian peaks + linear baseline to each time point
- Generate fit reports and visualizations

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lmfit.models import GaussianModel, LinearModel

# Enable inline plotting
%matplotlib inline

## Load Data

Update the file path below to point to your processed spectral data:

In [ ]:
# Update this path to your data file
data_file = 'C:/Users/jake_/Desktop/HGD_nanospec/DTNB_5/hgd_R37S_DTNB_1_5_transmission.asc_pyspec/final_pyspec.csv'

# Load the processed data from spec_main.py
data = pd.read_csv(data_file, index_col=0)
print(f"Data shape: {data.shape}")
print("\nFirst few rows:")
display(data.head())

## Filter Wavelengths

Filter out wavelengths below 280 nm:

In [ ]:
# Filter out wavelengths below 280
data = data[data.index.astype(float) >= 280]
print(f"Filtered data shape: {data.shape}")
print(f"Wavelength range: {data.index.astype(float).min():.1f} - {data.index.astype(float).max():.1f} nm")

## Define Fitting Model

The model consists of:
- 3 Gaussian peaks (centered around 290, 330, and 412 nm)
- 1 Linear baseline

In [ ]:
def create_model():
    """Create the composite model with 3 Gaussians + linear baseline"""
    
    # First Gaussian model (330 nm peak)
    model1 = GaussianModel(prefix='peak1_')
    params1 = model1.make_params(
        amplitude=dict(value=0.2, min=0),
        center=dict(value=330),
        sigma=dict(value=1)
    )
    
    # Second Gaussian model (412 nm peak)
    model2 = GaussianModel(prefix='peak2_')
    params2 = model2.make_params(
        amplitude=dict(value=0.2, min=0),
        center=dict(value=412, min=410, max=420),
        sigma=dict(value=1)
    )
    
    # Linear model for baseline
    model3 = LinearModel(prefix='base_')
    params3 = model3.make_params(
        slope=dict(value=0),
        intercept=dict(value=0)
    )
    
    # Third Gaussian model (290 nm peak)
    model4 = GaussianModel(prefix='peak3_')
    params4 = model4.make_params(
        amplitude=dict(value=0.2, min=0),
        center=dict(value=290, min=280, max=300),
        sigma=dict(value=1, min=0.1, max=1.5)
    )
    
    # Combine the models and parameters
    model = model1 + model2 + model3 + model4
    params = params1 + params2 + params3 + params4
    
    return model, params

## Fit All Time Points

Loop through each time point and fit the model. After the first fit, constrain subsequent fits based on initial results.

In [ ]:
# Initialize variables to store the fitted parameters
initial_params = None
fit_results = []

# Loop through each time point and perform the fitting
for i, time_point in enumerate(data.columns):
    y = data[time_point].values  # Absorbance values at the current time point
    x = data.index.values.astype(float)  # Wavelength values
    
    # Create model and parameters
    model, params = create_model()
    
    if initial_params is not None:
        # Update parameters with tight constraints except for amplitude
        for name, param in initial_params.items():
            if 'amplitude' not in name:
                params[name].set(value=param.value, min=param.value * 0.95, max=param.value * 1.05)
            else:
                params[name].set(value=param.value, min=0)
    
    # Fit the model to the data
    out = model.fit(y, params, x=x, method='leastsq', max_nfev=10000)
    fit_results.append(out)
    
    if initial_params is None:
        # Store the initial fitted parameters
        initial_params = out.params
    
    # Print progress
    if (i + 1) % 10 == 0:
        print(f"Fitted {i + 1}/{len(data.columns)} time points")

print(f"\nCompleted fitting {len(fit_results)} time points")

## Visualize Fits

Plot the first spectrum with fitted components:

In [ ]:
# Plot the first spectrum for checking
if len(fit_results) > 0:
    i = 0
    time_point = data.columns[i]
    out = fit_results[i]
    x = data.index.values.astype(float)
    y = data[time_point].values
    comps = out.eval_components()
    
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, label=f'data at {time_point}', s=5, alpha=0.6)
    plt.plot(x, out.best_fit, label='best fit', color='red', linewidth=2)
    plt.plot(x, comps['peak1_'], label='peak1 (330nm)', linestyle='--')
    plt.plot(x, comps['peak2_'], label='peak2 (412nm)', linestyle='--')
    plt.plot(x, comps['peak3_'], label='peak3 (290nm)', linestyle='--')
    plt.plot(x, comps['base_'], label='baseline', linestyle=':')
    plt.legend()
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Absorbance')
    plt.title(f'Spectrum at {time_point}')
    plt.grid(True, alpha=0.3)
    plt.show()

## View Fit Report

Display the fit report for the first time point:

In [ ]:
if len(fit_results) > 0:
    print(f"Fit report for time point: {data.columns[0]}")
    print(fit_results[0].fit_report(min_correl=0.3))

## Save Fit Reports

Save all fit reports to a log file:

In [ ]:
# Write all fit reports to a log file
with open('fit_report.log', 'w') as f:
    for i, (time_point, out) in enumerate(zip(data.columns, fit_results)):
        f.write(f'\nSpectra time_point: {time_point}\n')
        f.write(out.fit_report(min_correl=0.3))
        f.write('\n' + '='*80 + '\n')

print(f"Fit reports saved to 'fit_report.log'")

## Plot Multiple Time Points

Visualize fits at different time intervals:

In [ ]:
# Plot every 5th spectrum
n_plots = min(6, len(fit_results))
step = max(1, len(fit_results) // n_plots)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, i in enumerate(range(0, len(fit_results), step)[:n_plots]):
    time_point = data.columns[i]
    out = fit_results[i]
    x = data.index.values.astype(float)
    y = data[time_point].values
    
    axes[idx].scatter(x, y, s=3, alpha=0.6)
    axes[idx].plot(x, out.best_fit, 'r-', linewidth=2)
    axes[idx].set_title(f'Time: {time_point}')
    axes[idx].set_xlabel('Wavelength (nm)')
    axes[idx].set_ylabel('Absorbance')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Extract Peak Parameters Over Time

Extract and plot how peak parameters change over time:

In [ ]:
# Extract peak amplitudes over time
times = data.columns.to_list()
peak1_amps = [fit.params['peak1_amplitude'].value for fit in fit_results]
peak2_amps = [fit.params['peak2_amplitude'].value for fit in fit_results]
peak3_amps = [fit.params['peak3_amplitude'].value for fit in fit_results]

# Try to extract numeric time values
try:
    times_numeric = [float(t.replace('s', '')) if isinstance(t, str) else float(t) for t in times]
except:
    times_numeric = list(range(len(times)))

plt.figure(figsize=(12, 6))
plt.plot(times_numeric, peak1_amps, 'o-', label='Peak 1 (330nm)', alpha=0.7)
plt.plot(times_numeric, peak2_amps, 's-', label='Peak 2 (412nm)', alpha=0.7)
plt.plot(times_numeric, peak3_amps, '^-', label='Peak 3 (290nm)', alpha=0.7)
plt.xlabel('Time')
plt.ylabel('Peak Amplitude')
plt.title('Peak Amplitudes Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()